<a href="https://colab.research.google.com/github/Jhanavi-v/dataviz-exercises-jhanavivenkatesh/blob/main/week04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [5]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [6]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [8]:
# Task 1
# YOUR CODE HERE
# Task 1: Scatter - GDP vs Life Expectancy (2000 vs 2007)
# ========================================================

# Step 1: Choose your continent (NOT Africa, which was the example)
# Options: Europe, Asia, Americas, Oceania
chosen_continent = 'Europe'  # Change this!

# Step 2: Filter to that continent and both years
df_scatter = df[
    (df['continent'] == chosen_continent) &
    (df['year'].isin([2002, 2007])) # Changed 2000 to 2002 as 2000 is not in the dataset
].copy()

print(f"Countries in {chosen_continent}: {df_scatter['country'].nunique()}")
print(df_scatter.head(10))

# Step 3: Create scatter chart
fig1 = px.scatter(
    df_scatter,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    size='pop',
    hover_name='country',
    hover_data={'year': True, 'gdpPercap': ':.0f', 'lifeExp': ':.1f', 'pop': ':.0f'},
    color_discrete_map={2002: '#95A5A6', 2007: '#3498DB'},  # Grey (2002) and Blue (2007)
    size_max=40,  # Control bubble size
    title=f'<b>{chosen_continent}: Progress or Stagnation? GDP & Life Expectancy 2002-2007</b>',
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'year': 'Year'
    }
)

# Step 4: Make GDP axis logarithmic
fig1.update_xaxes(type='log')

# Step 5: Add annotations to highlight change
# Find countries that made most progress
df_2002 = df_scatter[df_scatter['year'] == 2002].set_index('country') # Changed df_2000 to df_2002
df_2007 = df_scatter[df_scatter['year'] == 2007].set_index('country')

# Calculate life expectancy improvement
improvement = df_2007['lifeExp'] - df_2002['lifeExp'] # Changed df_2000 to df_2002
best_improver = improvement.idxmax()
best_improvement = improvement.max()

fig1.add_annotation(
    x=df_2007.loc[best_improver, 'gdpPercap'],
    y=df_2007.loc[best_improver, 'lifeExp'],
    text=f'<b>{best_improver}</b><br>+{best_improvement:.1f} years',
    showarrow=True,
    arrowsize=1.5,
    arrowcolor='#3498DB',
    font=dict(size=11, color='#3498DB'),
    bgcolor='rgba(255,255,255,0.8)',
    bordercolor='#3498DB',
    borderwidth=1
)

# Step 6: Polish the layout
fig1.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        gridcolor='#ECF0F1',
        zeroline=False,
        showgrid=True
    ),
    yaxis=dict(
        gridcolor='#ECF0F1',
        zeroline=False,
        showgrid=True
    ),
    hovermode='closest',
    margin=dict(l=100, r=100, t=120, b=80),
    font=dict(family='Arial', size=12, color='#2C3E50'),
    height=550,
    width=900,
    legend=dict(
        title='<b>Year</b>',
        x=0.02,
        y=0.98,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#95A5A6',
        borderwidth=1
    )
)

fig1.show()

Countries in Europe: 30
                    country continent  year  lifeExp       pop     gdpPercap  \
22                  Albania    Europe  2002   75.651   3508512   4604.211737   
23                  Albania    Europe  2007   76.423   3600523   5937.029526   
82                  Austria    Europe  2002   78.980   8148312  32417.607690   
83                  Austria    Europe  2007   79.829   8199783  36126.492700   
118                 Belgium    Europe  2002   78.320  10311970  30485.883750   
119                 Belgium    Europe  2007   79.441  10392226  33692.605080   
154  Bosnia and Herzegovina    Europe  2002   74.090   4165416   6018.975239   
155  Bosnia and Herzegovina    Europe  2007   74.852   4552198   7446.298803   
190                Bulgaria    Europe  2002   72.140   7661799   7696.777725   
191                Bulgaria    Europe  2007   73.005   7322858  10680.792820   

    iso_alpha  iso_num  
22        ALB        8  
23        ALB        8  
82        AUT       

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [10]:
import plotly.graph_objects as go

# Task 2
# YOUR CODE HERE
# Task 2: Bubble Chart - 2007 Global View (Grey + Highlight)
# ===========================================================

# Step 1: Filter to 2007
df_bubble = df[df['year'] == 2007].copy()

# Step 2: Choose continent to highlight (not Africa)
highlight_continent = 'Asia'  # Change this!

# Step 3: Create a color column (grey or highlight color)
def assign_color(row):
    if row['continent'] == highlight_continent:
        return '#E74C3C'  # Red for highlight
    else:
        return '#BDBDBD'  # Light grey for others

df_bubble['color'] = df_bubble.apply(assign_color, axis=1);

print(f"Highlighted continent: {highlight_continent}")
print(df_bubble[df_bubble['continent'] == highlight_continent][['country', 'gdpPercap', 'lifeExp', 'pop']].head())

# Step 4: Create bubble chart using go.Figure (for more control)
fig2 = go.Figure()

# Add grey bubbles first (so they're behind)
for continent in df_bubble['continent'].unique():
    if continent != highlight_continent:
        data = df_bubble[df_bubble['continent'] == continent]
        fig2.add_trace(go.Scatter(
            x=data['gdpPercap'],
            y=data['lifeExp'],
            mode='markers',
            name=continent,
            marker=dict(
                size=data['pop'] / 1000000,  # Scale population to bubble size
                sizemode='diameter',
                sizeref=2 * max(df_bubble['pop']) / (40 ** 2),
                sizemin=4,
                color='#BDBDBD',
                line=dict(width=0.5, color='white')
            ),
            text=data['country'],
            hovertemplate='<b>%{text}</b><br>GDP: $%{x:,.0f}<br>Life Exp: %{y:.1f} years<extra></extra>',
            showlegend=False
        ))

# Add highlighted continent (on top, bold color)
highlight_data = df_bubble[df_bubble['continent'] == highlight_continent]
fig2.add_trace(go.Scatter(
    x=highlight_data['gdpPercap'],
    y=highlight_data['lifeExp'],
    mode='markers',
    name=highlight_continent,
    marker=dict(
        size=highlight_data['pop'] / 1000000,
        sizemode='diameter',
        sizeref=2 * max(df_bubble['pop']) / (40 ** 2),
        sizemin=4,
        color='#E74C3C',
        line=dict(width=1, color='white')
    ),
    text=highlight_data['country'],
    hovertemplate='<b>%{text}</b><br>GDP: $%{x:,.0f}<br>Life Exp: %{y:.1f} years<br>Pop: millions<extra></extra>',
    showlegend=True
))

# Step 5: Add annotations for interesting points
# Find highest life expectancy
highest_lifeexp = df_bubble.loc[df_bubble['lifeExp'].idxmax()]
fig2.add_annotation(
    x=highest_lifeexp['gdpPercap'],
    y=highest_lifeexp['lifeExp'],
    text=f"<b>{highest_lifeexp['country']}</b><br>Longest life",
    showarrow=True,
    arrowsize=1.5,
    font=dict(size=11, color='#27AE60')
)

# Find lowest life expectancy
lowest_lifeexp = df_bubble.loc[df_bubble['lifeExp'].idxmin()]
fig2.add_annotation(
    x=lowest_lifeexp['gdpPercap'],
    y=lowest_lifeexp['lifeExp'],
    text=f"<b>{lowest_lifeexp['country']}</b><br>Shortest life",
    showarrow=True,
    arrowsize=1.5,
    font=dict(size=11, color='#E74C3C')
)

# Step 6: Update layout
fig2.update_layout(
    title={
        'text': f'<b>The Great Health-Wealth Divide: {highlight_continent} in Global Context (2007)</b><br><sub>GDP per capita vs life expectancy; bubble size = population</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#2C3E50'}
    },
    xaxis=dict(
        title='<b>GDP per Capita (USD, log scale)</b>',
        type='log',
        gridcolor='#ECF0F1',
        zeroline=False,
        showgrid=True
    ),
    yaxis=dict(
        title='<b>Life Expectancy (years)</b>',
        gridcolor='#ECF0F1',
        zeroline=False,
        showgrid=True,
        range=[30, 85]  # Set range for clarity
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    hovermode='closest',
    margin=dict(l=120, r=100, t=140, b=100),
    font=dict(family='Arial', size=12, color='#2C3E50'),
    height=650,
    width=1000,
    legend=dict(
        title=f'<b>Highlighted: {highlight_continent}</b>',
        x=0.02,
        y=0.98,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#E74C3C',
        borderwidth=2
    )
)

fig2.show()

Highlighted continent: Asia
         country     gdpPercap  lifeExp         pop
11   Afghanistan    974.580338   43.828    31889923
95       Bahrain  29796.048340   75.635      708573
107   Bangladesh   1391.253792   64.062   150448339
227     Cambodia   1713.778686   59.723    14131858
299        China   4959.114854   72.961  1318683096
